# processing large ensembles

In [1]:
import warnings
warnings.filterwarnings('ignore')

In [2]:
import numpy as np
import os
import xarray as xr
import xcdat as xc
import matplotlib.pyplot as plt
from matplotlib.colors import BoundaryNorm as BM
import pandas as pd
import matplotlib as mpl
import matplotlib.ticker as mticker
import netCDF4
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from cartopy.util import add_cyclic_point
from cartopy.mpl.gridliner import LONGITUDE_FORMATTER, LATITUDE_FORMATTER

In [3]:
from scipy import stats

In [4]:
mpl.rcParams['font.family'] = 'Droid Sans'
mpl.rcParams['font.size'] = 12
# Edit axes parameters
mpl.rcParams['axes.linewidth'] = 1.5
# Tick properties
mpl.rcParams['xtick.major.size'] = 5
mpl.rcParams['xtick.minor.size'] = 3
mpl.rcParams['xtick.major.width'] = 1
mpl.rcParams['xtick.direction'] = 'out'
mpl.rcParams['ytick.major.size'] = 5
mpl.rcParams['ytick.minor.size'] = 3
mpl.rcParams['ytick.major.width'] = 1
mpl.rcParams['ytick.direction'] = 'out'

### Functions needed for the analysis

In [5]:
import matplotlib as m
from matplotlib.colors import BoundaryNorm as BM
import matplotlib.patches as mpatches

def plot_background(ax):
    ax.add_feature(cfeature.COASTLINE, alpha=0.9, lw=1.1)
    ax.set_global()
    # ax.add_feature(cfeature.LAND, color='lightgray')
    # ax.add_feature(cfeature.OCEAN, color='lightgray')
    gl = ax.gridlines(draw_labels=True,
                      linewidth=1, color='gray', alpha=0.01, linestyle='--')
    gl.top_labels = False
    # gl.left_labels = False
    # gl.bottom_labels = False
    gl.right_labels = False
    gl.xlines = False
    # gl.xlocator = mticker.FixedLocator([-180, -45, 0, 45, 180])
    gl.xformatter = LONGITUDE_FORMATTER
    gl.yformatter = LATITUDE_FORMATTER
    gl.xlabel_style = {'size': 10, 'color': 'k'}
    gl.ylabel_style = {'size': 10, 'color': 'k'}
    return ax


def plot_maps(x, y, z, titles, labels, cmap, levels, cbar_label = 'Precip', pval = [], nrows=1, ncols=3, figsize=(12,4), land_mask_list = [0], add_patch=False):
    fig, axarr = plt.subplots(nrows=nrows, ncols=ncols, figsize=figsize, constrained_layout=True, subplot_kw={'projection':ccrs.Robinson(central_longitude=180)})
    
    axlist = axarr.flatten()
    
    for ax in axlist:
        plot_background(ax)
    
    for i in range(len(z)):
        axlist[i].contourf(x, y, z[i], cmap = cmap, transform = ccrs.PlateCarree(central_longitude=0), levels=levels, extend='both')
        axlist[i].set_title(titles[i])
        if i in land_mask_list:
            axlist[i].add_feature(cfeature.LAND, color = 'k', zorder=1)
        if pval != []:
            pval_plot = np.ma.masked_less_equal(pval[i], 0.05)
            axlist[i].pcolor(x, y, pval_plot, alpha = 0., hatch='////', transform = ccrs.PlateCarree(central_longitude=0))
        axlist[i].set_title(titles[i], fontdict={'fontsize':12})
        axlist[i].text(0.1, 1.05, labels[i], size=16, fontweight='bold', transform=axlist[i].transAxes)
        if add_patch:
            axlist[i].add_patch(mpatches.Rectangle(xy=[120, -65], width=170, height=20,
                                            facecolor='none', edgecolor='k',
                                            transform=ccrs.PlateCarree()))
            axlist[i].add_patch(mpatches.Rectangle(xy=[190, -5], width=80, height=10,
                                            facecolor='none', edgecolor='k',
                                            transform=ccrs.PlateCarree()))
            axlist[i].add_patch(mpatches.Rectangle(xy=[140, -5], width=30, height=10,
                                            facecolor='none', edgecolor='k',
                                            transform=ccrs.PlateCarree()))
            axlist[i].add_patch(mpatches.Rectangle(xy=[250, -30], width=40, height=20,
                                            facecolor='none', edgecolor='k',
                                            transform=ccrs.PlateCarree()))
        
    norm = BM(levels, 256, extend='both')
    fig.colorbar(m.cm.ScalarMappable(norm = norm, cmap=cmap), ax = axlist, \
                orientation = 'horizontal', shrink=0.4, aspect = 20, pad = 0.05, label = cbar_label)

In [6]:
from functions import preproc_funcs as funcs

In [7]:
from functions import xr_lowess

In [8]:
from statsmodels.tsa.seasonal import STL


def loess1d(x, period):
    x_copy = x.copy()
    res = STL(x_copy, period=period).fit()
    return res.trend


def loess3d(x, dim, period):
    return xr.apply_ufunc(loess1d, x, input_core_dims=[[dim]], output_core_dims=[[dim]], kwargs=dict(period=period), vectorize=True, dask="parallelized")

In [9]:
import glob
import multiprocessing as mp

In [27]:
# Function to find the first file in each model's r1* directory
def find_all_files(pattern):
    all_paths = glob.glob(pattern)
    model_files = {}
    for path in all_paths:
        # Adjust the split indices based on your folder structure
        path_parts = path.split('/')
        # Assuming the model name is at index 7 (adjust if needed)
        model_identifier = path_parts[7] + '_' + path_parts[9][:-1]
        # model_identifier = path_parts[8] + '_' + path_parts[10][:-1]
        if model_identifier not in model_files:
            model_files[model_identifier] = '/'.join(path_parts[:-1]) + '/*.nc'  # Store only the first file for each model
    return model_files


# Function to find the first file in each model's r1* directory
def find_all_files_extended(pattern):
    all_paths = glob.glob(pattern)
    model_files = {}
    for path in all_paths:
        # Adjust the split indices based on your folder structure
        path_parts = path.split('/')
        # Assuming the model name is at index 7 (adjust if needed)
        model_identifier = path_parts[8] + '_' + path_parts[10][:-1]
        if model_identifier not in model_files:
            model_files[model_identifier] = '/'.join(path_parts[:-1]) + '/*.nc'  # Store only the first file for each model
    return model_files



## find the files for a single model

In [28]:
model = 'ACCESS-ESM1-5'

In [29]:
ts_pattern_hist = f'/g/data/lp01/CMIP6/CMIP/*/{model}/historical/*/Amon/ts/gr1.5/*/*.nc'
ts_pattern_ssp5 = f'/g/data/lp01/CMIP6/ScenarioMIP/*/{model}/ssp585/*/Amon/ts/gr1.5/*/*.nc'
ts_pattern_ssp3 = f'/g/data/lp01/CMIP6/ScenarioMIP/*/{model}/ssp370/*/Amon/ts/gr1.5/*/*.nc'
ts_pattern_ssp2 = f'/g/data/lp01/CMIP6/ScenarioMIP/*/{model}/ssp245/*/Amon/ts/gr1.5/*/*.nc'
ts_pattern_ssp1 = f'/g/data/lp01/CMIP6/ScenarioMIP/*/{model}/ssp126/*/Amon/ts/gr1.5/*/*.nc'


In [30]:
pr_pattern_hist = f'/g/data/lp01/CMIP6/CMIP/*/{model}/historical/*/Amon/pr/gr1.5/*/*.nc'
pr_pattern_ssp5 = f'/g/data/lp01/CMIP6/ScenarioMIP/*/{model}/ssp585/*/Amon/pr/gr1.5/*/*.nc'
pr_pattern_ssp3 = f'/g/data/lp01/CMIP6/ScenarioMIP/*/{model}/ssp370/*/Amon/pr/gr1.5/*/*.nc'
pr_pattern_ssp2 = f'/g/data/lp01/CMIP6/ScenarioMIP/*/{model}/ssp245/*/Amon/pr/gr1.5/*/*.nc'
pr_pattern_ssp1 = f'/g/data/lp01/CMIP6/ScenarioMIP/*/{model}/ssp126/*/Amon/pr/gr1.5/*/*.nc'


In [14]:
ts_pattern_ssp5o_ext = f'/g/data/*/*/CMIP6/ScenarioMIP/*/{model}/ssp534-over/*/Amon/ts/gn/*/*.nc'

In [15]:
pr_pattern_ssp5o_ext = f'/g/data/*/*/CMIP6/ScenarioMIP/*/{model}/ssp534-over/*/Amon/pr/gn/*/*.nc'

In [16]:
psl_pattern_hist = f'/g/data/lp01/CMIP6/CMIP/*/{model}/historical/*/Amon/psl/gr1.5/*/*.nc'
psl_pattern_ssp5 = f'/g/data/lp01/CMIP6/ScenarioMIP/*/{model}/ssp585/*/Amon/psl/gr1.5/*/*.nc'
psl_pattern_ssp3 = f'/g/data/lp01/CMIP6/ScenarioMIP/*/{model}/ssp370/*/Amon/psl/gr1.5/*/*.nc'
psl_pattern_ssp2 = f'/g/data/lp01/CMIP6/ScenarioMIP/*/{model}/ssp245/*/Amon/psl/gr1.5/*/*.nc'
psl_pattern_ssp1 = f'/g/data/lp01/CMIP6/ScenarioMIP/*/{model}/ssp126/*/Amon/psl/gr1.5/*/*.nc'


In [31]:
ts_files_hist = find_all_files(ts_pattern_hist)
ts_files_ssp5 = find_all_files(ts_pattern_ssp5)
ts_files_ssp3 = find_all_files(ts_pattern_ssp3)
ts_files_ssp2 = find_all_files(ts_pattern_ssp2)
ts_files_ssp1 = find_all_files(ts_pattern_ssp1)

In [32]:
pr_files_hist = find_all_files(pr_pattern_hist)
pr_files_ssp5 = find_all_files(pr_pattern_ssp5)
pr_files_ssp3 = find_all_files(pr_pattern_ssp3)
pr_files_ssp2 = find_all_files(pr_pattern_ssp2)
pr_files_ssp1 = find_all_files(pr_pattern_ssp1)

In [19]:
ts_files_ssp5o_ext = find_all_files_extended(ts_pattern_ssp5o_ext)

In [20]:
pr_files_ssp5o_ext = find_all_files_extended(pr_pattern_ssp5o_ext)

In [22]:
psl_files_hist = find_all_files(psl_pattern_hist)
psl_files_ssp5 = find_all_files(psl_pattern_ssp5)
psl_files_ssp3 = find_all_files(psl_pattern_ssp3)
psl_files_ssp2 = find_all_files(psl_pattern_ssp2)
psl_files_ssp1 = find_all_files(psl_pattern_ssp1)

In [33]:
pr_files_ssp5

{'ACCESS-ESM1-5_r6i1p1f': '/g/data/lp01/CMIP6/ScenarioMIP/CSIRO/ACCESS-ESM1-5/ssp585/r6i1p1f1/Amon/pr/gr1.5/v20210318/*.nc',
 'ACCESS-ESM1-5_r11i1p1f': '/g/data/lp01/CMIP6/ScenarioMIP/CSIRO/ACCESS-ESM1-5/ssp585/r11i1p1f1/Amon/pr/gr1.5/v20210714/*.nc',
 'ACCESS-ESM1-5_r35i1p1f': '/g/data/lp01/CMIP6/ScenarioMIP/CSIRO/ACCESS-ESM1-5/ssp585/r35i1p1f1/Amon/pr/gr1.5/v20210831/*.nc',
 'ACCESS-ESM1-5_r18i1p1f': '/g/data/lp01/CMIP6/ScenarioMIP/CSIRO/ACCESS-ESM1-5/ssp585/r18i1p1f1/Amon/pr/gr1.5/v20210714/*.nc',
 'ACCESS-ESM1-5_r20i1p1f': '/g/data/lp01/CMIP6/ScenarioMIP/CSIRO/ACCESS-ESM1-5/ssp585/r20i1p1f1/Amon/pr/gr1.5/v20210714/*.nc',
 'ACCESS-ESM1-5_r2i1p1f': '/g/data/lp01/CMIP6/ScenarioMIP/CSIRO/ACCESS-ESM1-5/ssp585/r2i1p1f1/Amon/pr/gr1.5/v20210318/*.nc',
 'ACCESS-ESM1-5_r32i1p1f': '/g/data/lp01/CMIP6/ScenarioMIP/CSIRO/ACCESS-ESM1-5/ssp585/r32i1p1f1/Amon/pr/gr1.5/v20210831/*.nc',
 'ACCESS-ESM1-5_r34i1p1f': '/g/data/lp01/CMIP6/ScenarioMIP/CSIRO/ACCESS-ESM1-5/ssp585/r34i1p1f1/Amon/pr/gr1.5/v2021

In [34]:
import xesmf as xe

In [35]:
# temp_hist = xr.open_dataset(ts_files_hist['ACCESS-CM2'])
# temp_hist

In [36]:
ds_out = xe.util.cf_grid_2d(-0.75, 360, 1.5, -90, 90, 1.5)
ds_out

<xarray.Dataset>
Dimensions:             (lon: 240, bound: 2, lat: 120)
Coordinates:
  * lon                 (lon) float64 0.0 1.5 3.0 4.5 ... 355.5 357.0 358.5
  * lat                 (lat) float64 -89.25 -87.75 -86.25 ... 86.25 87.75 89.25
    latitude_longitude  float64 nan
Dimensions without coordinates: bound
Data variables:
    lon_bounds          (lon, bound) float64 -0.75 0.75 0.75 ... 357.8 359.2
    lat_bounds          (lat, bound) float64 -90.0 -88.5 -88.5 ... 88.5 90.0

In [37]:
from dask.diagnostics import ProgressBar

In [38]:
# Function to process a single model and return the detrended NINO3.4 and precip anomalies
def process_model(model_identifier):
    try:
        # print(f"Processing model: {model_identifier}")
        # Load datasets
        var_file_hist = pr_files_hist[model_identifier]
        var_file_ssp = pr_files_ssp5[model_identifier]
        ds_var_hist = xc.open_mfdataset(var_file_hist, use_cftime=True).sel(time = slice('1850-01-01', '2015-01-01'))
        ds_var_ssp = xc.open_mfdataset(var_file_ssp, use_cftime=True)
        # add custom time ranges
        ds_var_hist['time'] = xr.cftime_range('1850-01-01', '2015-01-01', freq='1M')
        ssp_end_year = int(ds_var_ssp.time.dt.year[-1])
        ds_var_ssp['time'] = xr.cftime_range('2015-01-01', f'{ssp_end_year + 1}-01-01', freq='1M')
        # combined = xr.concat([ds_var_hist, ds_var_ssp], dim='time')
        # regridder = xe.Regridder(combined, ds_out, 'bilinear', periodic=True, ignore_degenerate=True)
        #
        # with ProgressBar():
        # var = regridder(combined.pr.resample(time = 'AS-JUN').mean('time')).load()  # var data
        var = xr.concat([ds_var_hist, ds_var_ssp], dim='time').pr.resample(time = 'AS-JUN').mean('time').load()  # var data
        # precip = ds_precip['pr'] * 86400  # Convert kg/m²/s to mm/day

        # Calculate 3d values
        # var_anom = funcs.calc_anom_annual(var, var.sel(time = slice('1960', '1990')))
        # var_trend = funcs.calc_trend3d(var_anom.sel(time = slice('1980', '2014')), 'time')
        # var_trend_pval = funcs.calc_trend_pval3d(var_anom.sel(time = slice('1980', '2014')), 'time')

        # calc timeseries values
        # weights = np.cos(np.deg2rad(var.lat))

        # print(f'Completed: {model_identifie}')
        # return model_identifier, var_trend, var_trend_pval, gmst_anom, nino34_index, wp_var, ct_var, so_var
        return model_identifier, var
    except Exception as e:
        print(f"Error processing {model_identifier}: {e}")



# TODO : remapping required for ssp534_over
def process_model_overshoot(model_identifier):
    try:
        # print(f"Processing model: {model_identifier}")
        # Load datasets
        var_file_hist = ts_files_hist[model_identifier]
        var_file_ssp5_over = ts_files_ssp5o_ext[model_identifier]
        var_file_ssp5 = ts_files_ssp5[model_identifier]
        ds_var_hist = xr.open_mfdataset(var_file_hist, use_cftime=True).sel(time = slice('1850-01-01', '2015-01-01'))
        ds_var_ssp5_over = xr.open_mfdataset(var_file_ssp5_over, use_cftime=True)
        ds_var_ssp5 = xr.open_mfdataset(var_file_ssp5, use_cftime=True)
        # add custom time ranges
        ds_var_hist['time'] = xr.cftime_range('1850-01-01', '2015-01-01', freq='1M')
        ssp_start_year = int(ds_var_ssp5_over.time.dt.year[0])
        ssp_end_year = int(ds_var_ssp5_over.time.dt.year[-1])
        ds_var_ssp5 = ds_var_ssp5.sel(time = slice(str(2015), str(ssp_start_year-1)))
        ds_var_ssp5['time'] = xr.cftime_range('2015-01-01', f'{ssp_start_year}-01-01', freq='1M')
        ds_var_ssp5_over['time'] = xr.cftime_range(f'{ssp_start_year}-01-01', f'{ssp_end_year + 1}-01-01', freq='1M')
        regridder = xe.Regridder(ds_var_ssp5_over, ds_out, 'bilinear', periodic=True)
        ds_var_ssp5_over_regrid = regridder(ds_var_ssp5_over)
        combined = xr.concat([ds_var_hist, ds_var_ssp5, ds_var_ssp5_over_regrid], dim='time')
        
        var = regridder = (combined.ts.resample(time = 'AS-JUN').mean('time')).load()  # var data
        # precip = ds_precip['pr'] * 86400  # Convert kg/m²/s to mm/day

        # Calculate 3d values
        var_anom = funcs.calc_anom_annual(var, var.sel(time = slice('1960', '1990')))
        # # var_trend = funcs.calc_trend3d(var_anom.sel(time = slice('1980', '2014')), 'time')
        # var_trend_pval = funcs.calc_trend_pval3d(var_anom.sel(time = slice('1980', '2014')), 'time')

        # calc timeseries values
        # weights = np.cos(np.deg2rad(var.lat))
        # gmst_anom = var_anom.weighted(weights).mean(('lat', 'lon'))
        # nino34_index = funcs.detrend1d_check(var_anom.sel(lat = slice(-5,5), lon = slice(-170+360, -120+360)).weighted(weights).mean(('lat', 'lon')), period=15)
        # wp_var = var_anom.sel(lat = slice(-5,5), lon = slice(140, 170)).weighted(weights).mean(('lat', 'lon'))
        # ct_var = var_anom.sel(lat = slice(-5,5), lon = slice(190, 270)).weighted(weights).mean(('lat', 'lon'))
        # so_var = var_anom.sel(lat = slice(-65, -45), lon = slice(120, 290)).weighted(weights).mean(('lat', 'lon'))

        # print(f'Completed: {model_identifie}')
        return model_identifier, var_anom
    except Exception as e:
        print(f"Error processing {model_identifier}: {e}")
        # break


# 

In [39]:
models_to_process = [(model) for model in pr_files_hist if model in pr_files_ssp5]
models_to_process

['ACCESS-ESM1-5_r6i1p1f',
 'ACCESS-ESM1-5_r11i1p1f',
 'ACCESS-ESM1-5_r35i1p1f',
 'ACCESS-ESM1-5_r18i1p1f',
 'ACCESS-ESM1-5_r20i1p1f',
 'ACCESS-ESM1-5_r2i1p1f',
 'ACCESS-ESM1-5_r32i1p1f',
 'ACCESS-ESM1-5_r34i1p1f',
 'ACCESS-ESM1-5_r30i1p1f',
 'ACCESS-ESM1-5_r31i1p1f',
 'ACCESS-ESM1-5_r28i1p1f',
 'ACCESS-ESM1-5_r36i1p1f',
 'ACCESS-ESM1-5_r37i1p1f',
 'ACCESS-ESM1-5_r25i1p1f',
 'ACCESS-ESM1-5_r9i1p1f',
 'ACCESS-ESM1-5_r1i1p1f',
 'ACCESS-ESM1-5_r3i1p1f',
 'ACCESS-ESM1-5_r38i1p1f',
 'ACCESS-ESM1-5_r4i1p1f',
 'ACCESS-ESM1-5_r19i1p1f',
 'ACCESS-ESM1-5_r24i1p1f',
 'ACCESS-ESM1-5_r16i1p1f',
 'ACCESS-ESM1-5_r26i1p1f',
 'ACCESS-ESM1-5_r10i1p1f',
 'ACCESS-ESM1-5_r21i1p1f',
 'ACCESS-ESM1-5_r12i1p1f',
 'ACCESS-ESM1-5_r29i1p1f',
 'ACCESS-ESM1-5_r14i1p1f',
 'ACCESS-ESM1-5_r13i1p1f',
 'ACCESS-ESM1-5_r7i1p1f',
 'ACCESS-ESM1-5_r39i1p1f',
 'ACCESS-ESM1-5_r5i1p1f',
 'ACCESS-ESM1-5_r8i1p1f',
 'ACCESS-ESM1-5_r15i1p1f',
 'ACCESS-ESM1-5_r17i1p1f',
 'ACCESS-ESM1-5_r23i1p1f',
 'ACCESS-ESM1-5_r22i1p1f',
 'ACCESS-E

In [40]:
# Run multiprocessing and gather results
res_arr = []
with mp.Pool(processes=mp.cpu_count()) as pool:
    i = 0
    for res in pool.imap(process_model, models_to_process):
        res_arr.append(res)
        print(f'Completed {i+1}/{len(models_to_process)}', end='\r')
        i += 1



In [41]:
model_list = np.array(res_arr)[:, 0]
model_list

array(['ACCESS-ESM1-5_r6i1p1f', 'ACCESS-ESM1-5_r11i1p1f',
       'ACCESS-ESM1-5_r35i1p1f', 'ACCESS-ESM1-5_r18i1p1f',
       'ACCESS-ESM1-5_r20i1p1f', 'ACCESS-ESM1-5_r2i1p1f',
       'ACCESS-ESM1-5_r32i1p1f', 'ACCESS-ESM1-5_r34i1p1f',
       'ACCESS-ESM1-5_r30i1p1f', 'ACCESS-ESM1-5_r31i1p1f',
       'ACCESS-ESM1-5_r28i1p1f', 'ACCESS-ESM1-5_r36i1p1f',
       'ACCESS-ESM1-5_r37i1p1f', 'ACCESS-ESM1-5_r25i1p1f',
       'ACCESS-ESM1-5_r9i1p1f', 'ACCESS-ESM1-5_r1i1p1f',
       'ACCESS-ESM1-5_r3i1p1f', 'ACCESS-ESM1-5_r38i1p1f',
       'ACCESS-ESM1-5_r4i1p1f', 'ACCESS-ESM1-5_r19i1p1f',
       'ACCESS-ESM1-5_r24i1p1f', 'ACCESS-ESM1-5_r16i1p1f',
       'ACCESS-ESM1-5_r26i1p1f', 'ACCESS-ESM1-5_r10i1p1f',
       'ACCESS-ESM1-5_r21i1p1f', 'ACCESS-ESM1-5_r12i1p1f',
       'ACCESS-ESM1-5_r29i1p1f', 'ACCESS-ESM1-5_r14i1p1f',
       'ACCESS-ESM1-5_r13i1p1f', 'ACCESS-ESM1-5_r7i1p1f',
       'ACCESS-ESM1-5_r39i1p1f', 'ACCESS-ESM1-5_r5i1p1f',
       'ACCESS-ESM1-5_r8i1p1f', 'ACCESS-ESM1-5_r15i1p1f',
      

In [42]:
model_var = xr.concat(np.array(res_arr)[:, 1], dim=model_list, coords='minimal', compat='override').rename(dict(concat_dim = 'model')).to_dataset(name = 'pr')

In [43]:
out = xr.merge([model_var])
out

<xarray.Dataset>
Dimensions:  (lon: 240, lat: 120, time: 452, model: 40)
Coordinates:
  * lon      (lon) float64 0.0 1.5 3.0 4.5 6.0 ... 352.5 354.0 355.5 357.0 358.5
  * lat      (lat) float64 -89.25 -87.75 -86.25 -84.75 ... 86.25 87.75 89.25
  * time     (time) object 1849-06-01 00:00:00 ... 2300-06-01 00:00:00
  * model    (model) object 'ACCESS-ESM1-5_r6i1p1f' ... 'ACCESS-ESM1-5_r27i1p1f'
Data variables:
    pr       (model, time, lat, lon) float32 2.78e-06 2.753e-06 ... nan nan

In [44]:
out.to_netcdf(f'/g/data/ob22/as8561/data/enso_trans_stable/sst_grad_res/lens/{model}_ssp_pr_original.nc')